In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
"""Conversion between the computational and Pauli-Liouville bases."""

from math import log2
from typing import Optional

import numpy as np
from numpy.typing import ArrayLike

from qibo.backends import Backend, _check_backend
from qibo.config import raise_error


def comp_basis_to_pauli(
    operator: ArrayLike,
    normalize: bool = False,
    backend: Optional[Backend] = None,
) -> ArrayLike:
    """Convert an operator from the computational basis to the Pauli-Liouville (PL) basis.

    The PL representation decomposes :math:`A` as a weighted sum of :math:`n`-qubit
    Pauli strings :math:`\\{P\\}` drawn from :math:`\\mathcal{P}_{n}`:

    .. math::
        A = \\sum_{P \\in \\mathcal{P}_{n}} \\alpha(P) \\, P, \\quad
        \\alpha(P) = \\frac{1}{2^{n}} \\operatorname{tr}\\!\\left(P^{\\dagger} A\\right) \\, .

    Pauli strings are returned as a flat vector of length :math:`4^{n}` in lexicographic
    :math:`\\{I, X, Y, Z\\}` order: for an :math:`n`-qubit string each qubit carries a
    label in :math:`\\{I{=}0,\\, X{=}1,\\, Y{=}2,\\, Z{=}3\\}`, and the flat index is
    the base-4 integer with qubit 0 as the least-significant digit.  This convention
    matches the ordering used throughout :mod:`qibo.quantum_info`.

    The algorithm is **Algorithm 1** of [1] and runs in
    :math:`\\mathcal{O}(N^{2} \\log N)` time with :math:`\\mathcal{O}(1)` additional
    memory via the Fast Walsh-Hadamard Transform (FWHT), where :math:`N = 2^{n}`.
    The three steps are:

    1. **XOR transform** — for every column :math:`q`, permute the rows by
       :math:`r \\mapsto r \\oplus q`:

       .. math::
           a_{r,\\,q} \\leftarrow a_{r \\oplus q,\\,q} \\, .

    2. **Fast Walsh-Hadamard transform** — apply the unnormalised transform
       :math:`H^{\\otimes n}` to every row, where
       :math:`H = \\bigl[\\begin{smallmatrix} 1 & 1 \\\\ 1 & -1 \\end{smallmatrix}\\bigr]`:

       .. math::
           a_{r,\\,s} \\leftarrow \\sum_{q=0}^{N-1} a_{r,\\,q}\\,
           \\bigl(H^{\\otimes n}\\bigr)_{q,\\,s} \\, .

    3. **Prefactors** — multiply element :math:`(r, s)` by

       .. math::
           \\frac{(-i)^{|r \\wedge s|}}{2^{n}} \\, ,

       where :math:`r \\wedge s` is the bitwise AND and :math:`|\\cdot|` is the
       Hamming weight.

    Args:
        operator (ArrayLike): square array of shape :math:`(N, N)` with :math:`N = 2^{n}`,
            :math:`n \\geq 1`.  Both Hermitian and general operators are supported.
        normalize (bool): if ``True``, the Pauli coefficients are further multiplied by
            :math:`\\sqrt{2^{n}}`.  Under this scaling the output vector has unit
            Euclidean norm whenever ``operator`` is a pure-state density matrix, and
            the map :math:`\\rho \\mapsto \\sqrt{2^n}\\,\\alpha` is an isometry for the
            Hilbert-Schmidt inner product.  Defaults to ``False``.
        backend (:class:`qibo.backends.abstract.Backend`, optional): backend to be used
            in the execution.  If ``None``, it uses the current backend.
            Defaults to ``None``.

    Returns:
        ArrayLike: complex array of shape :math:`(4^{n},)` with the Pauli coefficients
        in lexicographic IXYZ order.  For Hermitian ``operator`` all entries are real
        up to floating-point rounding (Corollary 1.1 of [1]).

    Raises:
        TypeError: if ``operator`` does not have exactly two dimensions.
        ValueError: if ``operator`` is not square or its dimension is not a
            positive power of 2.

    Examples:
        >>> import numpy as np
        >>> from qibo.quantum_info.basis import comp_basis_to_pauli
        >>> comp_basis_to_pauli(np.eye(2, dtype=complex))
        array([1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j])
        >>> X = np.array([[0, 1], [1, 0]], dtype=complex)
        >>> comp_basis_to_pauli(X)
        array([0.+0.j, 1.+0.j, 0.+0.j, 0.+0.j])

    References:
        1. T. N. Georges, B. K. Berntson, C. Sünderhauf, A. V. Ivanov,
           *Pauli Decomposition via the Fast Walsh-Hadamard Transform*,
           `arXiv:2408.06206 <https://arxiv.org/abs/2408.06206>`__ (2024).
    """
    backend = _check_backend(backend)

    if len(operator.shape) != 2:
        raise_error(
            TypeError,
            f"``operator`` must be a 2-dimensional array, "
            f"but has {len(operator.shape)} dimension(s).",
        )

    if operator.shape[0] != operator.shape[1]:
        raise_error(
            ValueError,
            f"``operator`` must be a square array, "
            f"but has shape {operator.shape}.",
        )

    N = int(operator.shape[0])
    _n = log2(N)

    if N == 0 or _n != int(_n):
        raise_error(
            ValueError,
            f"The dimension of ``operator`` must be a positive power of 2, "
            f"but got {N}.",
        )

    n = int(_n)

    # ------------------------------------------------------------------
    # Work in NumPy throughout the algorithmic steps.
    # The algorithm is entirely composed of index permutations, additions,
    # subtractions, and element-wise multiplications — all of which translate
    # trivially across backends.  We cast to the target backend at the end.
    # ------------------------------------------------------------------
    B = np.array(backend.cast(operator, dtype=complex))

    # Step 1: XOR transform — permute rows of each column q by r -> r XOR q
    row_idx = np.arange(N, dtype=np.int64)
    for q in range(N):
        B[:, q] = B[row_idx ^ q, q]

    # Step 2: Fast Walsh-Hadamard Transform on each row (unnormalised)
    _fwht_rows_inplace(B)

    # Step 3: Element-wise prefactors  (-i)^|r & s| / N
    B *= _phase_prefactors(n, forward=True) / N

    # Reindex: (r, s) matrix -> flat IXYZ vector of length 4^n
    perm = _rs_to_pauli_perm(n)
    pl = np.empty(4**n, dtype=complex)
    pl[perm.ravel()] = B.ravel()

    if normalize:
        pl *= np.sqrt(N)

    return backend.cast(pl, dtype=complex)


def pauli_basis_to_comp(
    pauli_op: ArrayLike,
    normalize: bool = False,
    backend: Optional[Backend] = None,
) -> ArrayLike:
    """Convert an operator from the Pauli-Liouville basis to the computational basis.

    Reconstructs the matrix :math:`A` from its Pauli-Liouville coefficients
    :math:`\\alpha(P)` via:

    .. math::
        A = \\sum_{P \\in \\mathcal{P}_{n}} \\alpha(P) \\, P \\, .

    This function is the exact inverse of :func:`comp_basis_to_pauli` and implements
    **Eq. (9)** of [1], running Algorithm 1 in reverse:

    1. **Inverse prefactors** — multiply element :math:`(r, s)` by
       :math:`(+i)^{|r \\wedge s|} \\cdot 2^{n}`.

    2. **Inverse Fast Walsh-Hadamard transform** — apply :math:`H^{\\otimes n}`
       again (the FWHT is self-inverse up to a factor of :math:`2^{n}`) and divide
       every element by :math:`2^{n}`.

    3. **Inverse XOR transform** — the XOR column permutation is its own inverse,
       so applying it once more recovers the original column order.

    Args:
        pauli_op (ArrayLike): complex array of shape :math:`(4^{n},)` containing the
            Pauli coefficients in lexicographic IXYZ order (I=0, X=1, Y=2, Z=3 per
            qubit, qubit 0 as least-significant digit).
        normalize (bool): if ``True``, the output matrix is divided by
            :math:`\\sqrt{2^{n}}`, consistent with the normalisation applied in
            :func:`comp_basis_to_pauli`.  Defaults to ``False``.
        backend (:class:`qibo.backends.abstract.Backend`, optional): backend to be used
            in the execution.  If ``None``, it uses the current backend.
            Defaults to ``None``.

    Returns:
        ArrayLike: complex array of shape :math:`(2^{n}, 2^{n})`.

    Raises:
        TypeError: if ``pauli_op`` does not have exactly one dimension.
        ValueError: if the length of ``pauli_op`` is not a power of 4.

    Examples:
        >>> import numpy as np
        >>> from qibo.quantum_info.basis import pauli_basis_to_comp
        >>> pl = np.array([0, 1, 0, 0], dtype=complex)   # X
        >>> pauli_basis_to_comp(pl)
        array([[0.+0.j, 1.+0.j],
               [1.+0.j, 0.+0.j]])

    References:
        1. T. N. Georges, B. K. Berntson, C. Sünderhauf, A. V. Ivanov,
           *Pauli Decomposition via the Fast Walsh-Hadamard Transform*,
           `arXiv:2408.06206 <https://arxiv.org/abs/2408.06206>`__ (2024).
    """
    backend = _check_backend(backend)

    if len(pauli_op.shape) != 1:
        raise_error(
            TypeError,
            f"``pauli_op`` must be a 1-dimensional array, "
            f"but has {len(pauli_op.shape)} dimension(s).",
        )

    M = int(pauli_op.shape[0])
    log4 = log2(M) / 2
    n = int(round(log4))

    if 4**n != M:
        raise_error(
            ValueError,
            f"The length of ``pauli_op`` must be a power of 4, "
            f"but got {M}.",
        )

    N = 2**n

    # Map the flat IXYZ vector back to the (N, N) coefficient matrix
    perm = _rs_to_pauli_perm(n)
    pl_np = np.array(backend.cast(pauli_op, dtype=complex))
    B = pl_np[perm]  # shape (N, N)

    # Reverse Step 3: multiply by (+i)^|r & s| * N
    B *= _phase_prefactors(n, forward=False) * N

    # Reverse Step 2: inverse FWHT = FWHT / N
    _fwht_rows_inplace(B)
    B /= N

    # Reverse Step 1: XOR permutation is its own inverse
    row_idx = np.arange(N, dtype=np.int64)
    for q in range(N):
        B[:, q] = B[row_idx ^ q, q]

    if normalize:
        B /= np.sqrt(N)

    return backend.cast(B, dtype=complex)


def _fwht_rows_inplace(B: np.ndarray) -> None:
    """Apply the Fast Walsh-Hadamard Transform to every row of *B* in-place.

    Uses the standard Cooley-Tukey butterfly.  The transform is unnormalised, i.e.
    it computes :math:`B[r, :] \\leftarrow H^{\\otimes n} \\cdot B[r, :]` where
    :math:`H = \\bigl[\\begin{smallmatrix} 1 & 1 \\\\ 1 & -1 \\end{smallmatrix}\\bigr]`.

    Parameters
    ----------
    B:
        2-D complex NumPy array whose number of columns is a power of 2.
        Modified in-place.
    """
    N = B.shape[1]
    h = 1
    while h < N:
        for start in range(0, N, 2 * h):
            u = B[:, start : start + h].copy()
            v = B[:, start + h : start + 2 * h].copy()
            B[:, start : start + h] = u + v
            B[:, start + h : start + 2 * h] = u - v
        h *= 2


def _rs_to_pauli_perm(n: int) -> np.ndarray:
    """Build the ``(2^n, 2^n)`` integer permutation that maps the paper's ``(r, s)``
    coefficient matrix to a flat IXYZ Pauli index.

    For each bit position :math:`j` the pair :math:`(r_j, s_j)` selects a
    single-qubit Pauli label according to:

    .. math::
        (0,0) \\to I = 0, \\quad
        (1,0) \\to X = 1, \\quad
        (1,1) \\to Y = 2, \\quad
        (0,1) \\to Z = 3 \\, .

    The flat IXYZ index is :math:`\\sum_{j=0}^{n-1} p_j \\cdot 4^{j}` with qubit 0
    as the least-significant digit.

    Parameters
    ----------
    n:
        Number of qubits.

    Returns
    -------
    numpy.ndarray of shape :math:`(2^n, 2^n)` and dtype ``int64``.
    ``perm[r, s]`` is the flat IXYZ index of Pauli string :math:`P_{r,s}`.
    """
    N = 2**n
    # Encode (rj, sj) as rj*2 + sj; map encoded value -> Pauli label
    # 0 = (0,0) -> I=0 | 1 = (0,1) -> Z=3 | 2 = (1,0) -> X=1 | 3 = (1,1) -> Y=2
    _enc_to_pauli = np.array([0, 3, 1, 2], dtype=np.int64)

    r_arr = np.arange(N, dtype=np.int64)
    s_arr = np.arange(N, dtype=np.int64)
    perm = np.zeros((N, N), dtype=np.int64)

    for j in range(n):
        rj = (r_arr[:, None] >> j) & 1   # shape (N, 1)
        sj = (s_arr[None, :] >> j) & 1   # shape (1, N)
        pj = _enc_to_pauli[rj * 2 + sj]  # shape (N, N)
        perm += pj * (4**j)

    return perm


def _phase_prefactors(n: int, forward: bool) -> np.ndarray:
    r"""Compute the :math:`(\pm i)^{|r \wedge s|}` prefactor array.

    Parameters
    ----------
    n:
        Number of qubits; the returned array has shape :math:`(2^n, 2^n)`.
    forward:
        If ``True``, returns :math:`(-i)^{|r \wedge s|}` (used in
        :func:`comp_basis_to_pauli`).
        If ``False``, returns :math:`(+i)^{|r \wedge s|}` (used in
        :func:`pauli_basis_to_comp`).

    Returns
    -------
    numpy.ndarray of shape :math:`(2^n, 2^n)` and dtype ``complex128``.
    """
    N = 2**n
    r_idx = np.arange(N, dtype=np.int64)[:, None]
    s_idx = np.arange(N, dtype=np.int64)[None, :]
    and_rs = r_idx & s_idx
    hamming = np.zeros((N, N), dtype=np.int64)
    for bit in range(n):
        hamming += (and_rs >> bit) & 1
    base = (-1j) if forward else (1j)
    return base**hamming

In [4]:
"""Tests for qibo.quantum_info.basis — comp_basis_to_pauli and pauli_basis_to_comp.

Tests cover:

* Single-qubit Pauli matrices (I, X, Y, Z) as reference cases.
* Multi-qubit tensor-product Pauli strings with known IXYZ indices.
* Round-trip fidelity for random general, Hermitian, real-symmetric, and
  complex-symmetric operators.
* Corollaries 1.1, 1.3, 1.4 of [1] (reality / sparsity of coefficients for
  structured inputs).
* The ``normalize`` flag and its effect on round-trips.
* Input validation / error handling (TypeError, ValueError).
* Linearity over the complex field.
* Trace / normalization identities (Parseval identity for the Hilbert-Schmidt
  inner product).
* Explicit LCU reconstruction :math:`A = \\sum_P \\alpha(P) P`.

Run with::

    pytest tests/quantum_info/test_pauli_basis.py -v

References:
    [1] T. N. Georges et al., arXiv:2408.06206 (2024).
"""

from __future__ import annotations

import math

import numpy as np
import pytest

from qibo.backends import NumpyBackend


# ---------------------------------------------------------------------------
# Shared fixtures
# ---------------------------------------------------------------------------

BACKEND = NumpyBackend()

# Single-qubit Pauli matrices  (I=0, X=1, Y=2, Z=3)
_I = np.eye(2, dtype=complex)
_X = np.array([[0, 1], [1, 0]], dtype=complex)
_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
_Z = np.array([[1, 0], [0, -1]], dtype=complex)
_PAULIS_1Q = [_I, _X, _Y, _Z]


def _kron_n(*mats: np.ndarray) -> np.ndarray:
    """Kronecker product of a sequence of matrices."""
    result = mats[0]
    for m in mats[1:]:
        result = np.kron(result, m)
    return result


def _rand_complex(N: int, seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    return rng.standard_normal((N, N)) + 1j * rng.standard_normal((N, N))


def _rand_hermitian(N: int, seed: int = 0) -> np.ndarray:
    A = _rand_complex(N, seed)
    return A + A.conj().T


def _rand_real_symmetric(N: int, seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    A = rng.standard_normal((N, N))
    return (A + A.T).astype(complex)


def _rand_complex_symmetric(N: int, seed: int = 0) -> np.ndarray:
    A = _rand_complex(N, seed)
    return A + A.T  # transpose, NOT conjugate transpose


def _pauli_ixyz_index(labels: list[int]) -> int:
    """Return the flat IXYZ index for a multi-qubit Pauli string.

    Parameters
    ----------
    labels:
        Per-qubit Pauli labels (0=I, 1=X, 2=Y, 3=Z) from *most* to *least*
        significant qubit; qubit 0 is the least-significant digit.
    """
    n = len(labels)
    idx = 0
    for i, lab in enumerate(labels):
        qubit = n - 1 - i  # qubit index (0 = LSB)
        idx += lab * (4**qubit)
    return idx


def _flat_rs_index(r: int, s: int, n: int) -> int:
    """Replicate the (r, s) -> flat IXYZ index map from the implementation."""
    k = 0
    for j in range(n):
        rj = (r >> j) & 1
        sj = (s >> j) & 1
        if rj == 0 and sj == 0:
            pj = 0
        elif rj == 1 and sj == 0:
            pj = 1
        elif rj == 1 and sj == 1:
            pj = 2
        else:
            pj = 3
        k += pj * (4**j)
    return k


def _build_all_pauli_matrices(n: int) -> list[tuple[int, np.ndarray]]:
    """Return ``(flat_ixyz_index, matrix)`` for every n-qubit Pauli string.

    Uses the definition :math:`P_{r,s} = \\bigotimes_{j=n-1}^{0} i^{r_j \\wedge s_j}
    X^{r_j} Z^{s_j}` from Theorem 1 of the reference.
    """
    N = 2**n
    result = []
    for r in range(N):
        for s in range(N):
            mat = np.array([[1.0 + 0j]])
            for j in range(n - 1, -1, -1):
                rj = (r >> j) & 1
                sj = (s >> j) & 1
                phase = (1j) ** (rj & sj)
                single = np.linalg.matrix_power(_X, rj) @ np.linalg.matrix_power(
                    _Z, sj
                )
                mat = np.kron(mat, phase * single)
            flat_idx = _flat_rs_index(r, s, n)
            result.append((flat_idx, mat))
    return result


# ---------------------------------------------------------------------------
# 1. Single-qubit Pauli matrices
# ---------------------------------------------------------------------------


class TestSingleQubitPaulis:
    """comp_basis_to_pauli(P) should return a unit PL vector at the IXYZ index."""

    @pytest.mark.parametrize("label,mat", enumerate(_PAULIS_1Q))
    def test_forward_single_pauli(self, label: int, mat: np.ndarray):
        pl = comp_basis_to_pauli(mat, backend=BACKEND)
        assert pl.shape == (4,)
        expected = np.zeros(4, dtype=complex)
        expected[label] = 1.0
        np.testing.assert_allclose(pl, expected, atol=1e-14)

    @pytest.mark.parametrize("label,mat", enumerate(_PAULIS_1Q))
    def test_inverse_single_pauli(self, label: int, mat: np.ndarray):
        pl = np.zeros(4, dtype=complex)
        pl[label] = 1.0
        A = pauli_basis_to_comp(pl, backend=BACKEND)
        np.testing.assert_allclose(A, mat, atol=1e-14)


# ---------------------------------------------------------------------------
# 2. Multi-qubit tensor-product Pauli strings
# ---------------------------------------------------------------------------


class TestMultiQubitPauliStrings:
    """Tensor products of 1-qubit Paulis have known flat IXYZ indices."""

    @pytest.mark.parametrize(
        "labels",
        [
            [0, 0],      # II
            [1, 0],      # XI  (X on qubit 1, I on qubit 0)
            [0, 1],      # IX
            [1, 1],      # XX
            [2, 3],      # YZ
            [3, 2],      # ZY
            [1, 2, 3],   # XYZ
            [3, 0, 1],   # ZIX
        ],
    )
    def test_pauli_string_roundtrip(self, labels: list[int]):
        n = len(labels)
        mat = _kron_n(*[_PAULIS_1Q[lab] for lab in labels])
        pl = comp_basis_to_pauli(mat, backend=BACKEND)

        assert pl.shape == (4**n,)
        expected_pl = np.zeros(4**n, dtype=complex)
        expected_pl[_pauli_ixyz_index(labels)] = 1.0
        np.testing.assert_allclose(pl, expected_pl, atol=1e-14)

        mat_rec = pauli_basis_to_comp(pl, backend=BACKEND)
        np.testing.assert_allclose(mat_rec, mat, atol=1e-14)


# ---------------------------------------------------------------------------
# 3. Round-trip fidelity
# ---------------------------------------------------------------------------


class TestRoundTrip:
    """comp_basis_to_pauli followed by pauli_basis_to_comp must be the identity."""

    @pytest.mark.parametrize("n", [1, 2, 3, 4])
    def test_random_complex(self, n: int):
        A = _rand_complex(2**n, seed=n)
        A_rec = pauli_basis_to_comp(comp_basis_to_pauli(A, backend=BACKEND), backend=BACKEND)
        np.testing.assert_allclose(A_rec, A, atol=1e-12)

    @pytest.mark.parametrize("n", [1, 2, 3])
    def test_random_hermitian(self, n: int):
        H = _rand_hermitian(2**n, seed=n + 10)
        H_rec = pauli_basis_to_comp(comp_basis_to_pauli(H, backend=BACKEND), backend=BACKEND)
        np.testing.assert_allclose(H_rec, H, atol=1e-12)

    @pytest.mark.parametrize("n", [1, 2, 3])
    def test_real_symmetric(self, n: int):
        S = _rand_real_symmetric(2**n, seed=n + 20)
        S_rec = pauli_basis_to_comp(comp_basis_to_pauli(S, backend=BACKEND), backend=BACKEND)
        np.testing.assert_allclose(S_rec, S, atol=1e-12)

    def test_zero_matrix(self):
        pl = comp_basis_to_pauli(np.zeros((4, 4), dtype=complex), backend=BACKEND)
        np.testing.assert_allclose(pl, 0.0, atol=1e-14)

    @pytest.mark.parametrize("n", [1, 2, 3, 4])
    def test_identity_n_qubits(self, n: int):
        """Identity matrix -> unit PL vector at index 0 (the all-I string)."""
        pl = comp_basis_to_pauli(np.eye(2**n, dtype=complex), backend=BACKEND)
        expected = np.zeros(4**n, dtype=complex)
        expected[0] = 1.0
        np.testing.assert_allclose(pl, expected, atol=1e-14)

    @pytest.mark.parametrize("n", [1, 2, 3, 4])
    def test_pauli_to_comp_to_pauli(self, n: int):
        """pauli_basis_to_comp(comp_basis_to_pauli(pl)) == pl."""
        rng = np.random.default_rng(n * 31)
        pl = rng.standard_normal(4**n) + 1j * rng.standard_normal(4**n)
        pl_rec = comp_basis_to_pauli(pauli_basis_to_comp(pl, backend=BACKEND), backend=BACKEND)
        np.testing.assert_allclose(pl_rec, pl, atol=1e-11)


# ---------------------------------------------------------------------------
# 4. Structural properties (Corollaries 1.1, 1.3, 1.4 of the reference)
# ---------------------------------------------------------------------------


class TestStructuralProperties:

    @pytest.mark.parametrize("n", [1, 2, 3])
    def test_corollary_1_1_hermitian_gives_real_coefficients(self, n: int):
        """Corollary 1.1: :math:`A^\\dagger = A \\Rightarrow \\alpha(P) \\in \\mathbb{R}`."""
        H = _rand_hermitian(2**n, seed=n + 100)
        pl = comp_basis_to_pauli(H, backend=BACKEND)
        np.testing.assert_allclose(
            np.imag(pl), 0.0, atol=1e-12,
            err_msg=f"Hermitian matrix has imaginary PL coefficients for n={n}",
        )

    @pytest.mark.parametrize("n", [1, 2, 3])
    def test_corollary_1_3_real_symmetric_sparsity(self, n: int):
        """Corollary 1.3: real-symmetric matrices have :math:`\\alpha(P)=0` when
        :math:`|r \\wedge s|` is odd."""
        N = 2**n
        S = _rand_real_symmetric(N, seed=n + 200)
        pl = comp_basis_to_pauli(S, backend=BACKEND)
        for r in range(N):
            for s in range(N):
                if bin(r & s).count("1") % 2 == 1:
                    flat_idx = _flat_rs_index(r, s, n)
                    assert abs(pl[flat_idx]) < 1e-12, (
                        f"Expected zero at (r={r}, s={s}), |r&s| odd; "
                        f"got {pl[flat_idx]:.3e}"
                    )

    @pytest.mark.parametrize("n", [1, 2, 3])
    def test_corollary_1_3_real_symmetric_real_coefficients(self, n: int):
        """Real-symmetric matrices are Hermitian, so their PL coefficients are real."""
        S = _rand_real_symmetric(2**n, seed=n + 300)
        pl = comp_basis_to_pauli(S, backend=BACKEND)
        np.testing.assert_allclose(np.imag(pl), 0.0, atol=1e-12)

    @pytest.mark.parametrize("n", [1, 2, 3])
    def test_corollary_1_4_complex_symmetric_sparsity(self, n: int):
        """Corollary 1.4: complex-symmetric matrices have :math:`\\alpha(P)=0`
        when :math:`|r \\wedge s|` is odd."""
        N = 2**n
        S = _rand_complex_symmetric(N, seed=n + 400)
        pl = comp_basis_to_pauli(S, backend=BACKEND)
        for r in range(N):
            for s in range(N):
                if bin(r & s).count("1") % 2 == 1:
                    flat_idx = _flat_rs_index(r, s, n)
                    assert abs(pl[flat_idx]) < 1e-12, (
                        f"Expected zero for complex-symmetric at (r={r}, s={s}); "
                        f"got {pl[flat_idx]:.3e}"
                    )

    @pytest.mark.parametrize("n", [2, 3])
    def test_corollary_1_3_sparsity_fraction(self, n: int):
        """Corollary 1.3: the fraction of zero entries is
        :math:`(2^n - 1) / 2^{n+1}`."""
        N = 2**n
        S = _rand_real_symmetric(N, seed=n + 500)
        pl = comp_basis_to_pauli(S, backend=BACKEND)
        n_zeros = int(np.sum(np.abs(pl) < 1e-12))
        expected_zeros = int((2**n - 1) / 2 ** (n + 1) * 4**n)
        assert n_zeros == expected_zeros, (
            f"n={n}: expected {expected_zeros} zero entries, got {n_zeros}"
        )


# ---------------------------------------------------------------------------
# 5. Normalize flag
# ---------------------------------------------------------------------------


class TestNormalizeFlag:

    @pytest.mark.parametrize("n", [1, 2, 3])
    def test_normalize_flag_forward(self, n: int):
        """``normalize=True`` multiplies all PL coefficients by :math:`\\sqrt{2^n}`."""
        N = 2**n
        A = _rand_complex(N, seed=n + 600)
        pl_raw = comp_basis_to_pauli(A, normalize=False, backend=BACKEND)
        pl_norm = comp_basis_to_pauli(A, normalize=True, backend=BACKEND)
        np.testing.assert_allclose(pl_norm, pl_raw * np.sqrt(N), atol=1e-14)

    @pytest.mark.parametrize("n", [1, 2, 3])
    def test_normalize_flag_inverse(self, n: int):
        """``normalize=True`` divides the reconstructed matrix by :math:`\\sqrt{2^n}`."""
        N = 2**n
        rng = np.random.default_rng(n + 700)
        pl = rng.standard_normal(4**n) + 1j * rng.standard_normal(4**n)
        A_raw = pauli_basis_to_comp(pl, normalize=False, backend=BACKEND)
        A_norm = pauli_basis_to_comp(pl, normalize=True, backend=BACKEND)
        np.testing.assert_allclose(A_norm, A_raw / np.sqrt(N), atol=1e-14)

    @pytest.mark.parametrize("n", [1, 2, 3])
    def test_normalize_round_trip(self, n: int):
        """Round-trip with matching ``normalize`` values on both ends recovers the input."""
        N = 2**n
        A = _rand_complex(N, seed=n + 800)
        # normalize=True in comp_basis_to_pauli multiplies by sqrt(N);
        # normalize=True in pauli_basis_to_comp divides by sqrt(N): net effect is identity.
        pl = comp_basis_to_pauli(A, normalize=True, backend=BACKEND)
        A_rec = pauli_basis_to_comp(pl, normalize=True, backend=BACKEND)
        np.testing.assert_allclose(A_rec, A, atol=1e-12)

    @pytest.mark.parametrize("n", [1, 2])
    def test_pure_state_unit_norm(self, n: int):
        """For a pure-state density matrix with ``normalize=True``, the PL vector
        has unit Euclidean norm (Parseval / isometry property).

        For a pure state :math:`\\rho = |\\psi\\rangle\\langle\\psi|`:

        .. math::
            \\sum_P |\\sqrt{2^n}\\,\\alpha(P)|^2
            = 2^n \\cdot \\frac{\\operatorname{tr}(\\rho^2)}{2^n}
            = \\operatorname{tr}(\\rho^2) = 1 \\, .
        """
        N = 2**n
        # Build a random pure state
        rng = np.random.default_rng(n + 900)
        psi = rng.standard_normal(N) + 1j * rng.standard_normal(N)
        psi /= np.linalg.norm(psi)
        rho = np.outer(psi, psi.conj())
        pl = comp_basis_to_pauli(rho, normalize=True, backend=BACKEND)
        np.testing.assert_allclose(np.linalg.norm(pl), 1.0, atol=1e-12)


# ---------------------------------------------------------------------------
# 6. Linearity
# ---------------------------------------------------------------------------


class TestLinearity:

    def test_linearity_forward(self):
        """comp_basis_to_pauli is linear: ``f(aA + bB) == a f(A) + b f(B)``."""
        N = 4
        A = _rand_complex(N, seed=1)
        B = _rand_complex(N, seed=2)
        a, b = 2.5 - 1j, -0.3 + 0.7j
        pl_sum = comp_basis_to_pauli(a * A + b * B, backend=BACKEND)
        pl_lin = a * comp_basis_to_pauli(A, backend=BACKEND) + b * comp_basis_to_pauli(B, backend=BACKEND)
        np.testing.assert_allclose(pl_sum, pl_lin, atol=1e-12)

    def test_linearity_inverse(self):
        """pauli_basis_to_comp is linear: ``g(a pl_A + b pl_B) == a g(pl_A) + b g(pl_B)``."""
        n = 2
        rng = np.random.default_rng(99)
        pl_A = rng.standard_normal(4**n) + 1j * rng.standard_normal(4**n)
        pl_B = rng.standard_normal(4**n) + 1j * rng.standard_normal(4**n)
        a, b = 1.2 + 0.4j, -0.8
        A_sum = pauli_basis_to_comp(a * pl_A + b * pl_B, backend=BACKEND)
        A_lin = a * pauli_basis_to_comp(pl_A, backend=BACKEND) + b * pauli_basis_to_comp(pl_B, backend=BACKEND)
        np.testing.assert_allclose(A_sum, A_lin, atol=1e-12)


# ---------------------------------------------------------------------------
# 7. Trace and Hilbert-Schmidt identities
# ---------------------------------------------------------------------------


class TestTraceAndHilbertSchmidt:

    @pytest.mark.parametrize("n", [1, 2, 3])
    def test_trace_equals_n_times_pl0(self, n: int):
        """Since :math:`P_{0} = I^{\\otimes n}`, we have
        :math:`\\operatorname{tr}(A) = 2^n \\, \\alpha(I^{\\otimes n})`."""
        N = 2**n
        A = _rand_complex(N, seed=n + 50)
        pl = comp_basis_to_pauli(A, backend=BACKEND)
        np.testing.assert_allclose(N * pl[0], np.trace(A), atol=1e-12)

    @pytest.mark.parametrize("n", [1, 2, 3])
    def test_density_matrix_trace_one(self, n: int):
        """A valid density matrix has :math:`\\operatorname{tr}(\\rho) = 1`, hence
        :math:`\\alpha(I^{\\otimes n}) = 1 / 2^n`, i.e. ``N * pl[0].real == 1``."""
        N = 2**n
        H = _rand_hermitian(N, seed=n + 60)
        rho = H @ H.conj().T
        rho /= np.trace(rho)
        pl = comp_basis_to_pauli(rho, backend=BACKEND)
        np.testing.assert_allclose(N * pl[0].real, 1.0, atol=1e-12)

    def test_hilbert_schmidt_parseval(self):
        """Parseval identity for the Hilbert-Schmidt inner product:

        .. math::
            \\frac{\\operatorname{tr}(A^{\\dagger} B)}{2^n} =
            \\sum_P \\alpha_A(P)^* \\, \\alpha_B(P) \\, .
        """
        n, N = 2, 4
        A = _rand_complex(N, seed=77)
        B = _rand_complex(N, seed=88)
        pl_A = comp_basis_to_pauli(A, backend=BACKEND)
        pl_B = comp_basis_to_pauli(B, backend=BACKEND)
        hs_direct = np.trace(A.conj().T @ B) / N
        hs_pauli = np.dot(pl_A.conj(), pl_B)
        np.testing.assert_allclose(hs_pauli, hs_direct, atol=1e-12)


# ---------------------------------------------------------------------------
# 8. Explicit LCU reconstruction
# ---------------------------------------------------------------------------


class TestLCUReconstruction:
    """Verify :math:`A = \\sum_P \\alpha(P) P` using explicit Pauli matrices."""

    @pytest.mark.parametrize("n", [1, 2])
    def test_explicit_reconstruction(self, n: int):
        N = 2**n
        A = _rand_hermitian(N, seed=n + 99)
        pl = comp_basis_to_pauli(A, backend=BACKEND)
        all_paulis = _build_all_pauli_matrices(n)
        A_rec = sum(pl[idx] * mat for idx, mat in all_paulis)
        np.testing.assert_allclose(A_rec, A, atol=1e-12)


# ---------------------------------------------------------------------------
# 9. Input validation
# ---------------------------------------------------------------------------


class TestInputValidation:

    def test_comp_basis_to_pauli_1d_input_raises(self):
        with pytest.raises(Exception):
            comp_basis_to_pauli(np.ones(4, dtype=complex), backend=BACKEND)

    def test_comp_basis_to_pauli_3d_input_raises(self):
        with pytest.raises(Exception):
            comp_basis_to_pauli(np.ones((2, 2, 2), dtype=complex), backend=BACKEND)

    def test_comp_basis_to_pauli_non_square_raises(self):
        with pytest.raises(Exception):
            comp_basis_to_pauli(np.ones((2, 3), dtype=complex), backend=BACKEND)

    def test_comp_basis_to_pauli_not_power_of_two_raises(self):
        with pytest.raises(Exception):
            comp_basis_to_pauli(np.eye(3, dtype=complex), backend=BACKEND)

    def test_pauli_basis_to_comp_2d_input_raises(self):
        with pytest.raises(Exception):
            pauli_basis_to_comp(np.ones((2, 2), dtype=complex), backend=BACKEND)

    def test_pauli_basis_to_comp_wrong_length_raises(self):
        with pytest.raises(Exception):
            pauli_basis_to_comp(np.ones(6, dtype=complex), backend=BACKEND)

    def test_comp_basis_to_pauli_accepts_real_input(self):
        """Real-valued arrays should be silently cast to complex."""
        pl = comp_basis_to_pauli(np.eye(2, dtype=float), backend=BACKEND)
        assert np.iscomplexobj(pl)
        expected = np.zeros(4, dtype=complex)
        expected[0] = 1.0
        np.testing.assert_allclose(pl, expected, atol=1e-14)

    def test_pauli_basis_to_comp_accepts_real_input(self):
        """Real-valued PL arrays should be silently cast to complex."""
        A = pauli_basis_to_comp(np.array([1.0, 0.0, 0.0, 0.0]), backend=BACKEND)
        np.testing.assert_allclose(A, np.eye(2, dtype=complex), atol=1e-14)

    def test_comp_basis_to_pauli_n1_shape(self):
        """n=1 input produces a length-4 output."""
        pl = comp_basis_to_pauli(np.eye(2, dtype=complex), backend=BACKEND)
        assert pl.shape == (4,)

    def test_pauli_basis_to_comp_n1_shape(self):
        """Length-4 PL input produces a (2, 2) output."""
        A = pauli_basis_to_comp(np.array([1, 0, 0, 0], dtype=complex), backend=BACKEND)
        assert A.shape == (2, 2)

In [ ]:
import qibo
qibo.set_backend("numpy")
backend = qibo.get_backend()
from qibo.models.encodings import _ehrlich_codewords_up_to_k
from scipy.special import comb

In [ ]:
nqubits = 4
up_to_k = 1
list(_ehrlich_codewords_up_to_k(up_to_k, False, nqubits, backend))

In [ ]:
sum(int(comb(nqubits, weight)) for weight in range(up_to_k+1))

In [ ]:
from qibo.models.encodings import _ehrlich_algorithm

In [ ]:
def _ehrlich_codewords_up_to_k____2(
    up2k: int,
    nqubits: int | None = None,
    reversed_list: bool = False,
    backend=None,
):
    """
    Yield all bitstrings of length ``nqubits`` (or ``up2k`` if ``nqubits=None``)
    with Hamming weight from 0 to ``up2k`` (or reverse), such that consecutive
    strings have Hamming distance ≤ 2.
    """

    # Determine bitstring length
    n = up2k if nqubits is None else nqubits

    if up2k > n:
        raise ValueError("up2k must be <= nqubits.")

    if n == 0:
        yield ""
        return

    def ones_left(k: int):
        return "1" * k + "0" * (n - k)

    def ones_right(k: int):
        return "0" * (n - k) + "1" * k

    from qibo.quantum_info.utils import hamming_distance

    # Starting boundary (weight 0 or weight up2k)
    if reversed_list:
        last_emitted = ones_left(up2k)
    else:
        last_emitted = "0" * n

    yield last_emitted

    # Define weight iteration (exclude starting boundary)
    if reversed_list:
        weights = range(up2k - 1, -1, -1)
    else:
        weights = range(1, up2k + 1)

    for k in weights:
        # Skip boundary already yielded
        if (not reversed_list and k == 0) or (
            reversed_list and k == up2k
        ):
            continue

        left = ones_left(k)
        right = ones_right(k)

        if hamming_distance(last_emitted, left) <= hamming_distance(
            last_emitted, right
        ):
            initial = left
        else:
            initial = right

        k_seq = _ehrlich_algorithm(
            backend.cast([int(b) for b in initial[::-1]], dtype=backend.int64),
            False,
        )

        if reversed_list:
            yield from reversed(k_seq)
            last_emitted = k_seq[0]
        else:
            yield from k_seq
            last_emitted = k_seq[-1]


In [ ]:
nqubits = None
up_to_k = 1
list(_ehrlich_codewords_up_to_k____2(up_to_k, nqubits, False, backend))

In [ ]:
def print_state_pretty(
    state,
    print_per_excitation_block=True,
    print_probs=True,
    only_return_data=False,
    print_hw_info=True,
):

    stuff_to_print = []

    nqubits = int(np.log2(state.shape[0]))

    hw_list = []

    real_amps = np.all(np.isreal(state))

    indices = np.where(~np.isclose(np.abs(state), 0))[0]
    if print_per_excitation_block:
        indices = indices[::-1]

    if print_probs or real_amps:
        amp_width = 14
    else:
        amp_width = 18
    pos_width = 10
    bitstring_width = nqubits + 6
    hw_width = 5

    # stuff_to_print.append("\n")
    if print_probs:
        stuff_to_print.append(
            f"{'Prob':<{amp_width}} {'Position':<{pos_width}} {'Bitstring':<{bitstring_width}} {'HW':<{hw_width}}"
        )
    else:
        stuff_to_print.append(
            f"{'Amp':<{amp_width}} {'Position':<{pos_width}} {'Bitstring':<{bitstring_width}} {'HW':<{hw_width}}"
        )
    stuff_to_print.append("-" * (amp_width + pos_width + bitstring_width + hw_width))

    amps = []
    bitstrings = []
    for index in indices:
        bitstring = format(index, f"0{nqubits}b")
        hw = bitstring.count("1")

        if hw not in hw_list:
            hw_list.append(hw)

        if print_probs:
            stuff_to_print.append(
                f"{np.abs(state[index])**2:<{amp_width}.8f} {index:<{pos_width}} {bitstring:<{bitstring_width}} {hw:<{hw_width}}"
            )
        else:
            if real_amps:
                stuff_to_print.append(
                    f"{state[index].real:<{amp_width}.8f} {index:<{pos_width}} {bitstring:<{bitstring_width}} {hw:<{hw_width}}"
                )
            else:
                stuff_to_print.append(
                    f"{state[index]:<{amp_width}.4f} {index:<{pos_width}} {bitstring:<{bitstring_width}} {hw:<{hw_width}}"
                )

        amps.append(state[index].real if real_amps else state[index])
        bitstrings.append(bitstring)

    if not only_return_data:
        stuff_to_print.append("\n")
        print(*stuff_to_print, sep="\n")
        if print_hw_info:
            print(
                f"- State is a superposition of basis states of hamming weight: {hw_list}\n\n{'~'*40}"
            )

    info_amps_states = {
        "amp": amps,
        "bitstring": bitstrings,
    }

    return info_amps_states

In [ ]:
import numpy as np
from scipy.special import binom
from qibo.models.encodings import up_to_k_hamming_weight_encoder

complex_data, custom_codewords, keep_antictrls = False, False, False

nqubits = 4
up_to_k = 1

seed = 10
dim = sum(int(binom(nqubits, weight)) for weight in range(up_to_k+1))
rng = np.random.default_rng(seed)
data = rng.random(dim)
if complex_data:
    data = data.astype(complex) + 1j * rng.random(dim)
data /= np.linalg.norm(data)
codewords = backend.arange(dim) if custom_codewords else None

circuit = up_to_k_hamming_weight_encoder(
    data,
    nqubits=nqubits,
    up_to_k=up_to_k,
    codewords=codewords,
    keep_antictrls=keep_antictrls,
    backend=backend,
)
state = backend.execute_circuit(circuit).state()

In [ ]:
codewords = list(_ehrlich_codewords_up_to_k(up_to_k, False, nqubits, backend=backend))
codewords

In [ ]:
[i for i sorted(codewords)

In [ ]:
data

In [ ]:
state

In [ ]:
circuit.draw()

In [ ]:
_ = print_state_pretty(
    state,
    print_per_excitation_block=False,
    print_probs=True,
    only_return_data=False,
    print_hw_info=True,
)
